# mccoy public API: quickstart

mccoy is a personal Spotify listening journal. Its read-only JSON
API serves four GET endpoints. There's no key or auth. Each IP gets
about 60 requests a minute; past that you get a `429` with `Retry-After`.

- Docs: `/api/docs`. Spec: `/api/openapi.json` (OpenAPI 3.1)
- Responses carrying listening data include a `source` field: `live`
  or `demo` (deterministic sample data)

You need `requests`, `pandas` and `plotly`
(`pip install requests pandas plotly`). Set `MCCOY_API` to point at
another deployment, such as a local `http://127.0.0.1:8050`.

In [ ]:
import os

import pandas as pd
import plotly.express as px
import requests

BASE = os.environ.get(
    "MCCOY_API", "https://web-production-bee9a.up.railway.app"
)


def get(path, **params):
    """GET a JSON endpoint; raise on any HTTP error."""
    resp = requests.get(f"{BASE}{path}", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

## 1. The hard-bop musician network

Nodes are musicians, and an edge means two of them share a release.
The data comes from MusicBrainz and Discogs, not from anyone's
listening.

In [ ]:
graph = get("/api/v1/network/graph")
nodes = pd.DataFrame(graph["nodes"])
edges = pd.DataFrame(graph["edges"])
print(f"{len(nodes)} musicians, {len(edges)} connections")
display(nodes.sort_values("degree", ascending=False).head(10))

## 2. Six degrees: the shortest path between two musicians

When several paths tie on length, the one with the strongest links
wins. Each hop names one shared release.

In [ ]:
top = nodes.sort_values("degree", ascending=False)["id"].astype(str)
names = dict(zip(nodes["id"].astype(str), nodes["name"]))
a = top.iloc[0]
# The graph isn't fully connected, so walk up from the least-connected
# musicians until one is reachable (a few calls, well under the limit).
route = None
for b in top.iloc[::-1].head(10):
    if b == a:
        continue
    route = get("/api/v1/network/path", **{"from": a, "to": b})
    if route["path"] is not None:
        break
if route is None or route["path"] is None:
    print(f"No path found from {names[a]} in the first few tries")
else:
    for hop in route["hops"]:
        print(f"{hop['from']} -> {hop['to']}  ({hop['release']})")


## 3. Top-artist trends

Weekly snapshots of the top artists for a time window (`short_term`,
`medium_term` or `long_term`). The API returns sample data until at
least two snapshots exist.

In [ ]:
trends = get("/api/v1/trends/artists", time_range="short_term")
print("source:", trends["source"])
ranks = pd.DataFrame([
    {"captured_at": s["captured_at"], "artist": a["name"],
     "rank": a["rank"]}
    for s in trends["snapshots"]
    for a in s["artists"]
], columns=["captured_at", "artist", "rank"])
latest = ranks[ranks["captured_at"] == ranks["captured_at"].max()]
top10 = latest.nsmallest(10, "rank")["artist"]
fig = px.line(
    ranks[ranks["artist"].isin(top10)],
    x="captured_at", y="rank", color="artist", markers=True,
    title="Top-artist rank by week",
)
fig.update_yaxes(autorange="reversed")
fig.show()

## 4. When the listening happens: hour of week

The public endpoint always returns sample data. Real listening hours
stay private.

In [ ]:
how = get("/api/v1/listening/hour-of-week")
print("source:", how["source"])
cells = pd.DataFrame(how["hour_of_week"])
grid = cells.pivot_table(
    index="iso_dow", columns="hour", values="plays", fill_value=0
).reindex(index=range(1, 8), columns=range(24), fill_value=0)
grid.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
px.imshow(
    grid, aspect="auto", color_continuous_scale="Reds",
    labels={"x": "hour", "y": "", "color": "plays"},
    title="Plays by hour of week",
).show()

## 5. Errors

Errors come back as JSON `{"error": "..."}`: `400` for bad
parameters, `404` for an unknown musician id, `429` past the rate
limit (check `Retry-After`).

In [ ]:
resp = requests.get(
    f"{BASE}/api/v1/network/path", params={"from": a}, timeout=30
)
is_json = resp.headers.get("Content-Type", "").startswith(
    "application/json"
)
print(resp.status_code, resp.json() if is_json else resp.text[:200])